# Fragrantica Exploration and Cleaning

## Objective
Explore, assess, and clean the Fragrantica reference dataset obtained from Kaggle to prepare it for downstream fragrance analytics and marketplace product matching.

## Source Data
- **Source:** Kaggle Fragrantica dataset
- **Format:** CSV
- **Location:** `/Volumes/main/fragrance_analytics/reference_data/fra_perfumes.csv`
- **Working DataFrame:** `df_perfumes`

The original CSV file remains unchanged. All transformations are applied to the working DataFrame, and the processed dataset is stored separately.

## Workflow

### 1. Exploratory Data Analysis
- Inspect the schema, sample records, row counts, missing values, and column distributions.
- Investigate exact duplicate records and repeated fragrance URLs.
- Identify data quality issues requiring cleaning.

### 2. Data Cleaning and Standardization
- Standardize column names to lowercase `snake_case`.
- Convert `rating_value` to `double`.
- Convert `rating_count` to `long`, removing thousands separators and handling missing or invalid values.
- Remove exact duplicate rows.
- Remove duplicate records by `url`, retaining one record per fragrance URL.
- Generate a deterministic `id` from the fragrance URL.
- Validate the cleaned schema, identifiers, and final record counts.

### 3. Data Storage
Save the cleaned dataset as a Delta table, keeping the original CSV source unchanged.

## Output
**Delta table:** `main.fragrance_analytics.fragrantica_perfumes`

**Final schema:**
- `id` — string, deterministic identifier derived from the fragrance URL
- `name` — string
- `gender` — string
- `rating_value` — double
- `rating_count` — long
- `main_accords` — string
- `perfumers` — string
- `description` — string
- `url` — string

## Technical Stack
Databricks, Python, PySpark, Delta Lake.

## Downstream Use
The cleaned reference dataset supports fragrance accord analysis and matching relevant Fragrantica products to cleaned marketplace listings.

## Data Integrity Note
The source CSV remains immutable. Duplicate URLs are reduced to one record before IDs are generated. Deterministic identifiers are derived from the URL to remain consistent across notebook reruns.

In [0]:
# Inspect fra_perfumes.csv — header, schema, dimensions, and samples

csv_path_perfumes = "/Volumes/main/fragrance_analytics/reference_data/fra_perfumes.csv"

df_perfumes = spark.read.csv(csv_path_perfumes, header=True, inferSchema=True)

# Header (column names)
print("Header:")
print(df_perfumes.columns)

# Schema
print("\nSchema:")
df_perfumes.printSchema()

# Dimensions
print(f"Number of rows:    {df_perfumes.count():,}")
print(f"Number of columns: {len(df_perfumes.columns):,}")

# First 20 rows
print("\nFirst 20 rows:")
display(df_perfumes.limit(20))

In [0]:

from pyspark.sql import functions as F

# 1. Check missing values in each column
missing_counts = df_perfumes.select([
    F.count(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""),
            c
        )
    ).alias(c)
    for c in df_perfumes.columns
])

print("Missing values by column:")
display(missing_counts)

# 2. Inspect gender categories
print("Gender distribution:")
display(
    df_perfumes
    .groupBy("Gender")
    .count()
    .orderBy(F.desc("count"))
)

# 3. Check exact duplicate rows
total_rows = df_perfumes.count()
distinct_rows = df_perfumes.distinct().count()

print(f"Total rows:           {total_rows:,}")
print(f"Distinct rows:        {distinct_rows:,}")
print(f"Duplicate rows:       {total_rows - distinct_rows:,}")

# 4. Check duplicate URLs
url_stats = df_perfumes.agg(
    F.count("url").alias("non_null_urls"),
    F.countDistinct("url").alias("distinct_urls")
)

print("URL statistics:")
display(url_stats)

# 5. Inspect the Main Accords format
print("Sample Main Accords values:")
display(
    df_perfumes
    .select("Name", "Gender", "Main Accords")
    .limit(15)
)

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

source_columns = df_perfumes.columns

# 1. Find exact duplicate records across all source columns
duplicate_groups = (
    df_perfumes
    .groupBy(*source_columns)
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

print("Groups of exact duplicate records:")
display(duplicate_groups)

# 2. Show all rows belonging to exact duplicate groups, including those containing NULL values

source = df_perfumes.alias("src")
duplicates = duplicate_groups.select(*source_columns).alias("dup")

join_condition = F.lit(True)

for c in source_columns:
    join_condition = join_condition & (
        F.col(f"src.`{c}`").eqNullSafe(F.col(f"dup.`{c}`"))
    )

exact_duplicates = (
    source
    .join(duplicates, on=join_condition, how="inner")
    .select("src.*")
)

print("All rows in exact duplicate groups, including NULLs:")
display(exact_duplicates.orderBy("url"))

# 3. Inspect URLs associated with multiple records
duplicate_urls = (
    df_perfumes
    .groupBy("url")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("Name").alias("distinct_names"),
        F.countDistinct("Gender").alias("distinct_genders")
    )
    .filter(F.col("row_count") > 1)
    .orderBy(F.desc("row_count"))
)

print("URLs appearing more than once:")
display(duplicate_urls)


In [0]:
# Check data types of all columns
df_perfumes.printSchema()

In [0]:

from pyspark.sql import functions as F

# 1. Inspect distinct values in Rating Value
print("Distinct Rating Value entries:")
display(
    df_perfumes
    .groupBy("Rating Value")
    .count()
    .orderBy("Rating Value")
)

# 2. Inspect distinct values in Rating Count
print("Distinct Rating Count entries:")
display(
    df_perfumes
    .groupBy("Rating Count")
    .count()
    .orderBy("Rating Count")
)


In [0]:

from pyspark.sql import functions as F

# 1. Rename all columns to lowercase snake_case
df_perfumes = df_perfumes.toDF(
    "name",
    "gender",
    "rating_value",
    "rating_count",
    "main_accords",
    "perfumers",
    "description",
    "url"
)

# 2. Clean rating_value and convert to double
df_perfumes = df_perfumes.withColumn(
    "rating_value",
    F.when(
        F.trim(F.col("rating_value")).isin(
            "", "N/A", "n/a", "null"
        ),
        F.lit(None).cast("double")
    ).otherwise(
        F.trim(F.col("rating_value")).cast("double")
    )
)

# 3. Clean rating_count and convert to long
df_perfumes = df_perfumes.withColumn(
    "rating_count",
    F.when(
        F.trim(F.col("rating_count")).isin(
            "", "N/A", "n/a", "null"
        ),
        F.lit(None).cast("long")
    ).otherwise(
        F.regexp_replace(
            F.trim(F.col("rating_count")),
            ",",
            ""
        ).cast("long")
    )
)

df_perfumes.printSchema()

display(df_perfumes.limit(10))
print("Total rows:", df_perfumes.count())
print("Distinct URLs:", df_perfumes.select("url").distinct().count())
print("Exact duplicate rows:", df_perfumes.count() - df_perfumes.dropDuplicates().count())

In [0]:

from pyspark.sql import functions as F

# 1. Remove exact duplicate rows
df_perfumes = df_perfumes.dropDuplicates()

# 2. Keep only one row per URL
df_perfumes = df_perfumes.dropDuplicates(["url"])

# 3. Generate a deterministic ID from the URL
df_perfumes = df_perfumes.withColumn(
    "id",
    F.sha2(F.col("url"), 256)
)

# 4. Place ID as the first column
df_perfumes = df_perfumes.select(
    "id",
    "name",
    "gender",
    "rating_value",
    "rating_count",
    "main_accords",
    "perfumers",
    "description",
    "url"
)

# 5. Validate the final DataFrame
print("Final row count:", df_perfumes.count())
print("Distinct URL count:", df_perfumes.select("url").distinct().count())
print("Distinct ID count:", df_perfumes.select("id").distinct().count())
print("Null IDs:", df_perfumes.filter(F.col("id").isNull()).count())

df_perfumes.printSchema()

# 6. Replace the existing Delta table with the updated schema
(
    df_perfumes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.fragrance_analytics.fragrantica_perfumes")
)

print("Delta table recreated with the updated schema.")